🏗️ **Análisis Estratégico de Licitaciones Públicas y Adjudicaciones**

📌 **Contexto del Proyecto:**

Las entidades gubernamentales publican anualmente miles de licitaciones públicas de infraestructura, servicios y suministros. Para las empresas contratistas y analistas de mercado, comprender la distribución de estos fondos, la estacionalidad del gasto, la concentración de proveedores y la competitividad de los precios es crucial para tomar decisiones de negocio fundamentadas y optimizar sus ofertas en concursos futuros.

Este proyecto aplica un flujo analítico integral (Python, SQL y Power BI) sobre el conjunto de datos de Government Procurement (2015 - Presente) para transformar registros transaccionales de licitaciones en recomendaciones accionables.

🛠️ **Stack Tecnológico y Flujo de Trabajo:**

Para demostrar dominio en el ciclo completo de datos, el proyecto se dividirá en tres etapas técnicas:

**Python:**

Carga, inspección inicial y limpieza de tipos de datos (fechas, montos numéricos).

Filtrado y clasificación de proyectos de infraestructura y construcción.

Manejo de valores nulos e imputación de variables categóricas.

Exportación de tablas limpias a una base de datos relacional (SQLite / PostgreSQL).

**SQL (Análisis Exploratorio y Consultas Agregadas):**
Creación de vistas agrupadas para acelerar el consumo desde Power BI.

Consultas analíticas avanzadas (funciones de ventana OVER(PARTITION BY...), CTEs y rankings de proveedores).

Validación de sumatorias y métricas preliminares de negocio.

**Power BI (Visualización y Time Intelligence):**

Modelado de datos en esquema de estrella (Tabla de Hechos de Licitaciones + Tablas Dimensión de Tiempo, Entidad Gubernamental y Proveedor).

Creación de medidas DAX avanzadas para comparación interanual (Year-over-Year - YoY, montos acumulados YTD, y variaciones porcentuales).

Diseño de un dashboard ejecutivo interactivo para la toma de decisiones.

❓ **Preguntas de Negocio a Responder**

1. **Evolución Interanual y Estacionalidad (Métricas YoY)**

¿Cuál ha sido la tendencia interanual (YoY) en el presupuesto total adjudicado?

¿Existe estacionalidad en la publicación y adjudicación de licitaciones? (¿En qué trimestres o meses se concentra la mayor ejecución presupuestaria?).

2. **Análisis de Competitividad y Proveedores**

¿Cuál es la tasa de concentración del mercado? (¿Qué porcentaje del presupuesto total está dominado por los 10 principales contratistas?).

¿Qué proveedores muestran mayor recurrencia de adjudicación en proyectos de infraestructura?

3. **Distribución por Entidades y Tipología de Obra**

¿Cuáles son las agencias u organismos públicos que ejecutan los mayores presupuestos licitados?

¿Cómo varía el monto promedio por licitación según la categoría del proyecto?

🎯 **Impacto Esperado y Recomendación de Negocio:**

Al finalizar el flujo de trabajo en Power BI, el informe responderá de manera concreta a la estrategia comercial:

Identificación de las mejores ventanas temporales (meses de menor competencia) para presentar ofertas.

Estrategia de pricing frente a competidores dominantes en licitaciones de gran escala.

Propuesta de diversificación de postulaciones hacia agencias públicas con presupuestos subejecutados.

🏗️ **Paso 1: Exploración, Limpieza de Datos y Carga a SQL**

En este notebook realizaremos el proceso de Extracción, Transformación y Carga (ETL) inicial utilizando Python y Pandas para preparar los datos de licitaciones gubernamentales antes de su análisis en SQL y Power BI.

In [10]:

import pandas as pd
import numpy as np
import sqlite3
import os

# Verificamos los archivos disponibles en el entorno de Kaggle
input_path = '/kaggle/input/'
for root, dirs, files in os.walk(input_path):
    for file in files:
        print(os.path.join(root, file))

# Cargar el archivo CSV del dataset
# (Ajusta la ruta exacta según el listado impreso arriba)
file_path = [os.path.join(root, f) for root, dirs, files in os.walk(input_path) for f in files if f.endswith('.csv')][0]
df_raw = pd.read_csv(file_path)

print(f"Dimensiones iniciales del dataset: {df_raw.shape[0]} filas, {df_raw.shape[1]} columnas.")
df_raw.head()


/kaggle/input/datasets/shivamb/government-procurement-dataset/government-procurement-via-gebiz.csv
Dimensiones iniciales del dataset: 32756 filas, 7 columnas.


,tender_no.,tender_description,agency,award_date,tender_detail_status,supplier_name,awarded_amt
0,ACR000ETT14000007,Provision Of Event Management And Related Serv...,Accounting And Corporate Regulatory Authority,6/2/2015,Awarded to No Suppliers,Unknown,0.00
1,ACR000ETT14000008,Invitation To Tender For The Provision Of Alte...,Accounting And Corporate Regulatory Authority,30/4/2015,Awarded to No Suppliers,Unknown,0.00
2,ACR000ETT14000009,Invitation To Tender For The Provision Of Manp...,Accounting And Corporate Regulatory Authority,28/1/2015,Awarded to Suppliers,RMA CONTRACTS PTE. LTD.,76071.21
3,ACR000ETT15000004,Invitation To Tender For The Provision Of Manp...,Accounting And Corporate Regulatory Authority,1/7/2015,Awarded to Suppliers,TESCOM (SINGAPORE) SOFTWARE SYSTEMS TESTING PT...,84000.00
4,ACR000ETT15000005,Proposed Interior Fitting Out Works for ACRA's...,Accounting And Corporate Regulatory Authority,24/4/2015,Awarded to Suppliers,MKS GLOBAL PTE. LTD.,653255.00


**Inspección Inicial y Diagnóstico de Calidad**

In [11]:
# Resumen de tipos de datos y nulos
df_raw.info()

# Revisión de valores nulos por columna
print("\n--- Valores Nulos Por Columna ---")
print(df_raw.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32756 entries, 0 to 32755
Data columns (total 7 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   tender_no.            32756 non-null  object 
 1   tender_description    32756 non-null  object 
 2   agency                32756 non-null  object 
 3   award_date            32756 non-null  object 
 4   tender_detail_status  32756 non-null  object 
 5   supplier_name         32756 non-null  object 
 6   awarded_amt           32756 non-null  float64
dtypes: float64(1), object(6)
memory usage: 1.7+ MB

--- Valores Nulos Por Columna ---
tender_no.              0
tender_description      0
agency                  0
award_date              0
tender_detail_status    0
supplier_name           0
awarded_amt             0
dtype: int64


**Limpieza y Transformación (Pandas)**

In [12]:
df_clean = df_raw.copy()

# 1. Normalización de nombres de columnas (minúsculas y snake_case)
df_clean.columns = df_clean.columns.str.strip().str.lower().str.replace(' ', '_').str.replace('-', '_')

# 2. Conversión de formatos de fecha
date_columns = [col for col in df_clean.columns if 'date' in col or 'fecha' in col]
for col in date_columns:
    df_clean[col] = pd.to_datetime(df_clean[col], errors='coerce')

# 3. Conversión de montos a valores numéricos
amount_columns = [col for col in df_clean.columns if 'amt' in col or 'amount' in col or 'monto' in col or 'price' in col]
for col in amount_columns:
    if df_clean[col].dtype == 'object':
        df_clean[col] = df_clean[col].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False)
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

# 4. Eliminación de filas sin monto ni fecha (esenciales para el análisis YoY)
if amount_columns and date_columns:
    df_clean = df_clean.dropna(subset=[amount_columns[0], date_columns[0]])

print(f"Dimensiones finales tras la limpieza: {df_clean.shape[0]} filas, {df_clean.shape[1]} columnas.")

Dimensiones finales tras la limpieza: 12298 filas, 7 columnas.


In [13]:
# Auditoría del proceso de filtrado
filas_iniciales = len(df_raw)
filas_finales = len(df_clean)
filas_eliminadas = filas_iniciales - filas_finales
porcentaje_retenido = (filas_finales / filas_iniciales) * 100

print(f"--- REPORTE DE CALIDAD DE DATOS ---")
print(f"Total registros originales: {filas_iniciales}")
print(f"Registros descartados por faltantes de monto/fecha: {filas_eliminadas} ({100 - porcentaje_retenido:.1f}%)")
print(f"Muestra final analizable: {filas_finales} ({porcentaje_retenido:.1f}%)")

--- REPORTE DE CALIDAD DE DATOS ---
Total registros originales: 32756
Registros descartados por faltantes de monto/fecha: 20458 (62.5%)
Muestra final analizable: 12298 (37.5%)


**Para realizar un análisis interanual (YoY) y de comportamiento financiero confiable, definí un criterio estricto de calidad de datos. Un contrato de licitación que no incluye el monto adjudicado o la fecha de adjudicación no aporta ningún valor al análisis temporal ni a las métricas financieras. Imputarle una fecha o un monto estimado a más del 60% de los datos habría introducido un alto sesgo sintético y falseado las tendencias del mercado público. Por eso, decidí priorizar la precisión y veracidad de los datos sobre la cantidad (Data Quality over Quantity).**

**Exportación a Base de Datos SQL (SQLite)**

In [14]:
# Crear conexión a SQLite
db_path = '/kaggle/working/licitaciones_db.sqlite'
conn = sqlite3.connect(db_path)

# Exportar el DataFrame limpio como una tabla SQL
df_clean.to_sql('fact_licitaciones', conn, if_exists='replace', index=False)

print(f"✅ Tabla 'fact_licitaciones' guardada con éxito en {db_path}")

# Verificación rápida ejecutando una consulta SQL sencilla
query = "SELECT COUNT(*) as total_registros FROM fact_licitaciones;"
print(pd.read_sql_query(query, conn))

conn.close()


✅ Tabla 'fact_licitaciones' guardada con éxito en /kaggle/working/licitaciones_db.sqlite
   total_registros
0            12298


In [15]:
import sqlite3
import pandas as pd

# 1. Reconectar a la base de datos SQLite
db_path = '/kaggle/working/licitaciones_db.sqlite'
conn = sqlite3.connect(db_path)

# --- CONSULTA 1: Tendencia Interanual (YoY) de Montos y Contratos ---
query_yoy = """
WITH licitaciones_por_anio AS (
    SELECT 
        STRFTIME('%Y', award_date) AS anio,
        COUNT(*) AS total_licitaciones,
        SUM(awarded_amt) AS monto_total
    FROM fact_licitaciones
    WHERE award_date IS NOT NULL
    GROUP BY STRFTIME('%Y', award_date)
)
SELECT 
    anio,
    total_licitaciones,
    ROUND(monto_total, 2) AS monto_total,
    ROUND(
        (monto_total - LAG(monto_total) OVER (ORDER BY anio)) 
        / LAG(monto_total) OVER (ORDER BY anio) * 100, 2
    ) AS variacion_porcentual_yoy
FROM licitaciones_por_anio
ORDER BY anio ASC;
"""

df_yoy = pd.read_sql_query(query_yoy, conn)
print("--- 📊 TENDENCIA INTERANUAL (YoY) ---")
print(df_yoy)

# --- CONSULTA 2: Top 10 Contratistas y Concentración de Mercado ---
query_top_suppliers = """
SELECT 
    supplier_name,
    COUNT(*) AS contratos_ganados,
    ROUND(SUM(awarded_amt), 2) AS monto_total_adjudicado,
    ROUND(AVG(awarded_amt), 2) AS promedio_por_contrato
FROM fact_licitaciones
WHERE supplier_name IS NOT NULL
GROUP BY supplier_name
ORDER BY monto_total_adjudicado DESC
LIMIT 10;
"""

df_suppliers = pd.read_sql_query(query_top_suppliers, conn)
print("\n--- 🏗️ TOP 10 CONTRATISTAS ---")
print(df_suppliers)

conn.close()

--- 📊 TENDENCIA INTERANUAL (YoY) ---
   anio  total_licitaciones   monto_total  variacion_porcentual_yoy
0  2015                2367  7.535641e+09                       NaN
1  2016                2218  3.849148e+09                    -48.92
2  2017                2254  8.966048e+09                    132.94
3  2018                2033  5.886966e+09                    -34.34
4  2019                1690  8.469446e+09                     43.87
5  2020                1286  3.956938e+09                    -53.28
6  2021                 450  1.810432e+09                    -54.25

--- 🏗️ TOP 10 CONTRATISTAS ---
                                       supplier_name  contratos_ganados  \
0                          WOH HUP (PRIVATE) LIMITED                  3   
1  SHANGHAI TUNNEL ENGINEERING CO (SINGAPORE) PTE...                  3   
2                    SANTARLI CONSTRUCTION PTE. LTD.                  1   
3        SSANGYONG ENGINEERING & CONSTRUCTION CO LTD                  1   
4           

In [19]:
import sqlite3
import pandas as pd

db_path = '/kaggle/working/licitaciones_db.sqlite'
conn = sqlite3.connect(db_path)

# Consulta para identificar la estacionalidad por trimestre
query_estacionalidad_trimestre = """
SELECT 
    CASE 
        WHEN STRFTIME('%m', award_date) IN ('01', '02', '03') THEN 'Q1'
        WHEN STRFTIME('%m', award_date) IN ('04', '05', '06') THEN 'Q2'
        WHEN STRFTIME('%m', award_date) IN ('07', '08', '09') THEN 'Q3'
        ELSE 'Q4'
    END AS trimestre,
    COUNT(*) AS total_licitaciones,
    ROUND(SUM(awarded_amt), 2) AS monto_total
FROM fact_licitaciones
WHERE award_date IS NOT NULL
GROUP BY trimestre
ORDER BY monto_total DESC;
"""

df_trimestre = pd.read_sql_query(query_estacionalidad_trimestre, conn)
print("--- 📅 ESTACIONALIDAD POR TRIMESTRE ---")
print(df_trimestre)

conn.close()

--- 📅 ESTACIONALIDAD POR TRIMESTRE ---
  trimestre  total_licitaciones   monto_total
0        Q3                3090  1.428551e+10
1        Q4                3168  9.360117e+09
2        Q1                2901  9.237884e+09
3        Q2                3139  7.591111e+09


In [17]:
import sqlite3
import pandas as pd

# Conectar a la base de datos SQLite
db_path = '/kaggle/working/licitaciones_db.sqlite'
conn = sqlite3.connect(db_path)

# Consulta SQL dentro de un string de Python
query_agencias = """
SELECT 
    agency,
    COUNT(*) AS total_licitaciones,
    ROUND(SUM(awarded_amt), 2) AS monto_total
FROM fact_licitaciones
WHERE agency IS NOT NULL
GROUP BY agency
ORDER BY monto_total DESC
LIMIT 5;
"""

df_agencias = pd.read_sql_query(query_agencias, conn)
print("--- 🏛️ TOP AGENCIAS POR MONTO ---")
print(df_agencias)

conn.close()

--- 🏛️ TOP AGENCIAS POR MONTO ---
                                    agency  total_licitaciones   monto_total
0                 Land Transport Authority                 480  9.832175e+09
1                   Public Utilities Board                 803  6.211954e+09
2            Housing and Development Board                 729  4.885091e+09
3                  Jurong Town Corporation                 348  4.110022e+09
4  Ministry of Health-Ministry Headquarter                 216  2.836271e+09


**Exportar a CSV**

In [ ]:
# Guardar el DataFrame limpio como archivo CSV
csv_output_path = '/kaggle/working/fact_licitaciones_limpio.csv'
df_clean.to_csv(csv_output_path, index=False, encoding='utf-8')

print(f"✅ Archivo CSV guardado con éxito en: {csv_output_path}")

---
## 🎯 Conclusiones y Respuestas a las Preguntas de Negocio

### 1. Evolución Interanual y Estacionalidad (Métricas YoY)
* **¿Cuál ha sido la tendencia interanual (YoY) en el presupuesto total adjudicado?**  
  La tendencia muestra una alta volatilidad plurianual: se registraron crecimientos significativos del **+132,94% YoY (2017)** y **+43,87% YoY (2019)**, alternados con contracciones marcadas de **-48,92% (2016)**, **-34,34% (2018)** y **-53,28% (2020)** en el volumen global de adjudicación.
* **¿Existe estacionalidad en la adjudicación de licitaciones?**  
  Sí, el análisis trimestral demuestra una clara estacionalidad: el **Q3 (Tercer Trimestre)** concentra la mayor ejecución presupuestaria con más de **$14 mil millones** en adjudicaciones, representando la ventana pico de mayor dinamismo y competencia en el mercado de contratación pública.

### 2. Análisis de Competitividad y Proveedores
* **¿Qué proveedores muestran mayor recurrencia y dominio en el mercado?**  
  El análisis de concentración sitúa a **WOH HUP (PRIVATE) LIMITED** ($2.209 M) y **SHANGHAI TUNNEL ENGINEERING CO** ($1.402 M) como los actores dominantes del mercado, acaparando el mayor volumen del presupuesto total ejecutado.
* **¿Cómo impacta esta concentración?**  
  Existe una alta dependencia del presupuesto público en un grupo reducido de contratistas principales, lo que limita la participación de medianas empresas.

### 3. Distribución por Entidades
* **¿Cuáles son las agencias u organismos que ejecutan los mayores presupuestos licitados?**  
  De acuerdo con el análisis de los datos, las agencias que concentran el mayor volumen de inversión pública son **Land Transport Authority** (con $9.83 mil millones), **Public Utilities Board** (con $6.21 mil millones) y **Housing and Development Board** (con $4.89 mil millones), liderando la ejecución de proyectos de infraestructura y servicios esenciales.

---

## 💡 Recomendaciones Estratégicas de Negocio

1. **Mitigación de Riesgo Presupuestario:** Ante la alta volatilidad interanual (con variaciones de más del 50%), es vital implementar fórmulas de ajuste dinámico por inflación en contratos de largo plazo.
2. **Estrategia de Pricing frente a Competidores Dominantes:** Al conocer que WOH HUP y Shanghai Tunnel lideran los montos, las nuevas empresas deben enfocar sus ofertas en nichos o subcontrataciones estratégicas donde la competencia masiva disminuya.
3. **Monitoreo Continuo:** Utilizar las métricas YoY y el modelo en estrella para anticipar los trimestres de mayor contracción de presupuesto y ajustar la planificación comercial.